# SPEAR-Net v4 — Global artisanal-mining detection (Colab, T4)

**SPEAR-Net**: a lightweight, **spectral-prior-guided**, recall-optimized network for
detecting **artisanal & small-scale (illegal-prone) mining** in global Sentinel-2 imagery.

This notebook runs the whole v4 pipeline on a free **T4**:
clone → install → **fetch the 35.6 MB verified annotations** → fetch a **subset of 6-band
Sentinel-2** from Microsoft Planetary Computer (free, no login) → chip + cache to Drive →
build SPEAR-Net (PISP gate) → train → evaluate (per-class + area-stratified) → PISP
explainability → **leave-one-region-out** generalization.

Dataset: `SimonJasansky/mine-segmentation` (Zenodo 14195737) — 1,210 sites, verified
masks (P99.2/R95.7), with an explicit **artisanal vs industrial** label.

> **Runtime → Change runtime type → T4 GPU**, then run cells top to bottom.
> Re-run cell 2 anytime to pull the latest fixes.


## 1. Check GPU

In [ ]:
!nvidia-smi || echo "No GPU — set Runtime > Change runtime type > T4 GPU"
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())


## 2. Clone the repository  *(re-run to pull fixes)*

**Interns:** fork the repo first (github.com/prakhar443/illegal_mining → *Fork*), then set
`REPO_URL` to **your fork** so your work never touches the original. See
[`docs/TEAM_SETUP.md`](https://github.com/prakhar443/illegal_mining/blob/spearnet-colab/docs/TEAM_SETUP.md).


In [ ]:
import os, subprocess
# Owner uses the original; interns set this to their fork, e.g.
#   REPO_URL = "https://github.com/<your-username>/illegal_mining.git"
REPO_URL = "https://github.com/prakhar443/illegal_mining.git"
BRANCH   = "spearnet-colab"
REPO_DIR = "illegal_mining"
if not os.path.exists(REPO_DIR):
    if subprocess.run(["git","clone","--branch",BRANCH,REPO_URL,REPO_DIR]).returncode != 0:
        subprocess.run(["git","clone",REPO_URL,REPO_DIR], check=True)
else:
    subprocess.run(["git","-C",REPO_DIR,"fetch","origin",BRANCH])
    subprocess.run(["git","-C",REPO_DIR,"checkout",BRANCH])
    subprocess.run(["git","-C",REPO_DIR,"pull","origin",BRANCH])
%cd {REPO_DIR}
!git log --oneline -1


## 3. Install dependencies (~3 min)

Deep-learning + the geospatial stack that fetches Sentinel-2 from Planetary Computer.

In [ ]:
!pip install -q "timm>=0.9.12" "segmentation-models-pytorch>=0.3.3" ptflops scikit-learn pyyaml
!pip install -q pystac pystac-client planetary-computer stackstac rioxarray \
                geopandas rasterio shapely pyproj pyogrio tqdm
!pip install -q -e .
print("Done. If a binary import (rasterio/geopandas) fails: Runtime > Restart session, "
      "then re-run from cell 3.")


## 4. Mount Drive + set paths (fetch local, store one zip on Drive)

**Why this design:** writing thousands of chips straight to the Drive FUSE mount in a long
loop causes `ConnectionAbortedError` crashes. So we fetch to **fast local disk**
(`/content/chips`) and persist the whole dataset as **one zip on Drive**
(`spearnet_chips.zip`). Next session, we restore that single zip in seconds — no re-fetch,
and nothing huge on GitHub.


In [ ]:
import os, time
CHIPS_DIR = "/content/chips"                         # fast local disk (stable for writes)
os.makedirs(CHIPS_DIR, exist_ok=True)
DRIVE_ZIP = "/content/drive/MyDrive/spearnet_chips.zip"   # single-file persistent backup

def _mount(attempts=3):
    from google.colab import drive
    for i in range(attempts):
        try:
            drive.mount('/content/drive', force_remount=(i > 0)); return True
        except Exception as e:
            print(f"  Drive mount attempt {i+1} failed: {e}"); time.sleep(3*(i+1))
    return False

DRIVE_OK = _mount()
print("local chips ->", CHIPS_DIR, "| Drive zip ->", DRIVE_ZIP if DRIVE_OK else "(Drive unavailable)")


### 4b. Restore chips from Drive if already fetched

If you fetched in a previous session and packaged to Drive, this restores the whole dataset
locally in seconds — **skip cells 5–6 and jump to cell 7**. First time through, this is a
no-op and you continue to the fetch.


In [ ]:
import sys; sys.path.insert(0, "src")
from spearnet.data.fetch import restore_chips
if DRIVE_OK and os.path.exists(DRIVE_ZIP) and not os.path.exists(os.path.join(CHIPS_DIR, "manifest.csv")):
    restore_chips(DRIVE_ZIP, CHIPS_DIR)
    import glob
    print("restored chips:", len(glob.glob(f"{CHIPS_DIR}/*/*_img.tif")))
else:
    print("No restore needed (no Drive zip yet, or chips already present). Proceed to fetch.")


### 4c. (Optional) Restore from a public URL — works on ANY Google account

The Drive zip (4b) only restores on the account that fetched it. To reuse the dataset from
**any** account, upload `spearnet_chips.zip` once to a public host and set `DATA_URL`:
a **GitHub Release asset** (≤2 GB), a **Hugging Face dataset** file, or a shared-Drive
direct link. Then every account skips fetching and pulls the same dataset.


In [ ]:
from spearnet.data.fetch import restore_chips_from_url
DATA_URL = ""   # e.g. "https://github.com/prakhar443/illegal_mining/releases/download/chips-v1/spearnet_chips.zip"
if DATA_URL and not os.path.exists(os.path.join(CHIPS_DIR, "manifest.csv")):
    restore_chips_from_url(DATA_URL, CHIPS_DIR)
    import glob; print("restored chips:", len(glob.glob(f"{CHIPS_DIR}/*/*_img.tif")))
else:
    print("Skipped (no DATA_URL set, or chips already present).")


## 5. Inspect annotations (metadata only)

Resolves the real file via the **Zenodo API** (robust to version/filename changes),
downloads the 35.6 MB GeoPackage, and prints the split / mine-type / **artisanal-vs-
industrial** counts — where you confirm the ASM signal is real and trainable.


In [ ]:
import sys; sys.path.insert(0, "src")
from spearnet.data.fetch import download_annotations, print_summary

# Default resolves the file via the Zenodo API. If Zenodo is flaky, pass a direct link:
#   gpkg = download_annotations("mine_data", annot_url="https://zenodo.org/records/14195737/files/<exact_name>?download=1")
gpkg = download_annotations("mine_data")
print_summary(gpkg)


## 6. Fetch a Sentinel-2 subset → chips  *(one-time, cached to Drive)*

Pulls 6 bands (B,G,R,NIR,SWIR1,SWIR2) per tile from Planetary Computer, rasterizes the
verified masks, chips 2048→256, stores **uint16** GeoTIFFs + a `manifest.csv` carrying the
per-chip `scale` and `region`.

**Recommended 3-pass fetch** (each pass appends to the manifest; safe to re-run):
1. pilot (artisanal, tiny) to confirm everything writes,
2. all artisanal (the rare, precious class),
3. a capped industrial set for contrast.

Start with the pilot, check it writes, then raise the caps.


In [ ]:
from spearnet.data.fetch import fetch_dataset

# ---- pilot: a few artisanal tiles to confirm the fetch works ----
fetch_dataset(
    out_dir=CHIPS_DIR, annot_dir="mine_data",
    fetch_imagery=True,
    scale_filter="artisanal",
    subset_per_split=8,        # small pilot; raise later
    chip_size=256, keep_empty_frac=0.3,
)
print("\nPilot done. Inspect a chip below, then run the full passes.")


### 6b. Full fetch — RESUMABLE (just re-run this cell if the runtime crashes)

`resume=True` skips tiles already fetched/attempted, so each re-run **continues** instead
of restarting. If Colab crashes mid-fetch, simply run this cell again until it prints
`Done` with no remaining tiles. Chips go to **local disk** (stable), not Drive.


In [ ]:
# Pass 2: ALL artisanal tiles (the rare, precious class)
fetch_dataset(out_dir=CHIPS_DIR, fetch_imagery=True, scale_filter="artisanal",
              subset_per_split=None, chip_size=256, resume=True)
# Pass 3: capped industrial tiles for contrast / hard negatives (lower the cap if disk tight)
fetch_dataset(out_dir=CHIPS_DIR, fetch_imagery=True, scale_filter="industrial",
              subset_per_split=80, chip_size=256, resume=True)


### 6c. Package the fetched chips → one zip on Drive  *(do this once fetch is complete)*

Stores the whole dataset as a single file so future sessions restore it in seconds
(cell 4b) without re-fetching. Re-run after fetching more tiles to refresh the backup.


In [ ]:
from spearnet.data.fetch import package_chips
if DRIVE_OK:
    package_chips(CHIPS_DIR, DRIVE_ZIP)
    print("Backed up to Drive. Next session, cell 4b restores this automatically.")
else:
    print("Drive not mounted — re-run cell 4 to mount, then package.")


Sanity-check one fetched chip (6 bands) and its mask:

In [ ]:
import glob, rasterio, numpy as np, matplotlib.pyplot as plt
imgs = sorted(glob.glob(f"{CHIPS_DIR}/*/*_img.tif"))
print("chips written:", len(imgs))
if imgs:
    ip = imgs[len(imgs)//2]; mp = ip.replace("_img.tif", "_mask.tif")
    with rasterio.open(ip) as s: img = s.read()
    with rasterio.open(mp) as s: msk = s.read()[0]
    print("img", img.shape, "dtype", img.dtype, "| mask uniques", np.unique(msk))
    rgb = img[[2,1,0]].astype("float32")
    lo, hi = np.percentile(rgb, 2), np.percentile(rgb, 98)
    rgb = ((rgb - lo)/(hi - lo + 1e-6)).clip(0, 1)
    fig, ax = plt.subplots(1,2,figsize=(8,4))
    ax[0].imshow(rgb.transpose(1,2,0)); ax[0].set_title("RGB"); ax[1].imshow(msk); ax[1].set_title("mask")
    for a in ax: a.axis("off"); plt.show()


## 7. Config — task + how much data

`v4_binary` (mining detector, robust primary) or `v4_scale3` (artisanal vs industrial, the headline). RUN_MODE caps train chips so a run fits Colab time.

In [ ]:
import torch
from spearnet.config import load_config
from spearnet.utils import set_seed

CONFIG   = "configs/v4_scale3.yaml"   # or configs/v4_binary.yaml
RUN_MODE = "dev"                      # "smoke" | "dev" | "full"

cfg = load_config(CONFIG)
cfg.data.chips_root = CHIPS_DIR
cfg.data.manifest   = f"{CHIPS_DIR}/manifest.csv"
cfg.data.num_workers = 2
cfg.run.device = "cuda" if torch.cuda.is_available() else "cpu"

if RUN_MODE == "smoke":
    cfg.data.subset_train, cfg.data.subset_val, cfg.optim.epochs = 200, 80, 8
elif RUN_MODE == "dev":
    cfg.data.subset_train, cfg.data.subset_val, cfg.optim.epochs = None, None, 40
elif RUN_MODE == "full":
    cfg.data.subset_train, cfg.data.subset_val, cfg.optim.epochs = None, None, 60

# Persist checkpoints to Drive so a Colab disconnect doesn't lose a multi-hour run.
import os
cfg.run.out_dir = f"/content/drive/MyDrive/spearnet_runs/{os.path.basename(CONFIG).split('.')[0]}_{cfg.data.task}"
os.makedirs(cfg.run.out_dir, exist_ok=True)
# Auto-resume if a previous run left a checkpoint here.
_ckpt = os.path.join(cfg.run.out_dir, "last.pt")
cfg.run.resume = _ckpt if os.path.exists(_ckpt) else None

set_seed(cfg.run.seed)
print(f"RUN_MODE={RUN_MODE} | task={cfg.data.task} ({cfg.num_classes} classes) | "
      f"prior={cfg.model.prior_type} | bands={cfg.data.bands} | device={cfg.run.device}")
print(f"checkpoints -> {cfg.run.out_dir}" + (f"  (resuming from {_ckpt})" if cfg.run.resume else ""))


## 8. Build dataloaders

In [ ]:
from spearnet.data import build_dataloaders
loaders = build_dataloaders(cfg, splits=("train", "val"))
print("train batches:", len(loaders["train"]), "| val batches:", len(loaders["val"]))
batch = next(iter(loaders["train"]))
print({k: tuple(v.shape) for k, v in batch.items()})
print("mask classes in batch:", torch.unique(batch["mask"]).tolist())


## 9. Visualize a sample + the PISP spectral priors

NDVI (vegetation loss), MNDWI (ponds), NDTI (turbidity), BSI (bare soil/tailings).

In [ ]:
import matplotlib.pyplot as plt
from spearnet.data import compute_priors, prior_names
from spearnet.data.priors import band_index
from spearnet.utils.viz import colorize_mask, _to_hwc_uint8
import os, numpy as np
PAPER_FIG = "/content/drive/MyDrive/spearnet_runs/paper_figures"; os.makedirs(PAPER_FIG, exist_ok=True)

img = batch["image"][0]
idx = band_index(cfg.data.band_order)
priors = compute_priors(img.unsqueeze(0), cfg.model.prior_type, idx)[0].numpy()
names = prior_names(cfg.model.prior_type)
# per-index colormaps; each index is signed in [-1, 1] -> symmetric range + colourbar (G5)
cmaps = {"NDVI": "RdYlGn", "MNDWI": "YlGnBu", "NDTI": "viridis", "BSI": "inferno"}
fig, ax = plt.subplots(1, 6, figsize=(26, 4.8))
ax[0].imshow(_to_hwc_uint8(img)); ax[0].set_title("S2 RGB", fontsize=13); ax[0].axis("off")
ax[1].imshow(colorize_mask(batch["mask"][0].numpy())); ax[1].set_title("Ground truth", fontsize=13); ax[1].axis("off")
for i, name in enumerate(names):
    a = ax[i + 2]
    vmax = float(np.nanmax(np.abs(priors[i]))) or 1.0
    im = a.imshow(priors[i], cmap=cmaps.get(name.upper(), "viridis"), vmin=-vmax, vmax=vmax)
    a.set_title(name, fontsize=13); a.axis("off")
    cb = fig.colorbar(im, ax=a, fraction=0.046, pad=0.04); cb.ax.tick_params(labelsize=8)
plt.tight_layout()
fig.savefig(f"{PAPER_FIG}/priors.pdf", bbox_inches="tight", dpi=300)
fig.savefig(f"{PAPER_FIG}/priors.png", bbox_inches="tight", dpi=300)
plt.show()
print("saved -> priors.pdf/.png (with colourbars + ranges) in", PAPER_FIG)


## 10. Build SPEAR-Net + efficiency report

In [ ]:
from spearnet.models import build_model
from spearnet.utils import count_parameters, measure_efficiency
import json
model = build_model(cfg)
print("Parameters:", count_parameters(model))
print(json.dumps(measure_efficiency(model, cfg.data.image_size, cfg.run.device,
                                    in_chans=cfg.data.bands), indent=2))


## 11. Train  *(prints estimated class weights first)*

*Checkpoints save to Drive **every epoch** (`last.pt` for resume, `best.pt` on improvement). If Colab disconnects, just re-run cells 1–8 then this cell — cell 7 auto-resumes from `last.pt`, continuing from the last completed epoch.*

In [ ]:
from spearnet.engine import Trainer
trainer = Trainer(model, loaders, cfg)
summary = trainer.train()
print("best", cfg.run.save_best_metric, "=", summary["best_metric"])


## 12. Evaluate (per-class IoU + area-stratified recall)

In [ ]:
from spearnet.engine import evaluate
device = torch.device(cfg.run.device)
results = evaluate(model, loaders["val"], cfg, device, compute_area_stratified=True)
print(f"mIoU={results['miou']:.4f}  mF1={results['mean_f1']:.4f}  "
      f"mRecall={results['mean_recall']:.4f}  pixAcc={results['pixel_acc']:.4f}")
for cls, m in results["per_class"].items():
    print(f"  {cls:>12}: IoU={m['iou']:.3f}  recall={m['recall']:.3f}  support={m['support']}")
print("\nArea-stratified recall:", results.get("area_stratified_recall"))


## 13. Explainability — qualitative panel (RGB | GT | prediction | attention)

Saves `qualitative.pdf` for the paper (rows = mining chips, columns = S2 RGB / ground truth / SPEAR-Net prediction / PISP attention).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from spearnet.utils.viz import colorize_mask, _to_hwc_uint8, overlay_csp_attention
import os, torch, numpy as np
PAPER_FIG = "/content/drive/MyDrive/spearnet_runs/paper_figures"; os.makedirs(PAPER_FIG, exist_ok=True)

# Use the TEST split (matches the paper caption); build it if not already loaded.
ld = loaders["test"] if "test" in loaders else build_dataloaders(cfg, splits=("test",))["test"]
model.eval(); device = torch.device(cfg.run.device)

def fg_iou(pred, gt):
    p = (pred > 0); g = (gt > 0)
    u = (p | g).sum().item()
    return (p & g).sum().item() / u if u else float("nan")

cand = []  # (iou, image, gt, pred, attn)
with torch.no_grad():
    for vb in ld:
        out = model(vb["image"].to(device)); preds = out["logits"].argmax(1).cpu()
        attn = out.get("attn")
        for i in range(preds.shape[0]):
            if (vb["mask"][i] > 0).any():
                a = attn[i].cpu() if attn is not None else None
                cand.append((fg_iou(preds[i], vb["mask"][i]), vb["image"][i], vb["mask"][i], preds[i], a))
        if len(cand) >= 80: break
cand.sort(key=lambda t: t[0], reverse=True)
picks = (cand[:2] + cand[-2:]) if len(cand) >= 4 else cand   # 2 success (high IoU) + 2 failure (low IoU)
tags  = ["Success", "Success", "Failure", "Failure"][:len(picks)]

cols = ["S2 RGB", "Ground truth", "SPEAR-Net", "PISP attention"]
fig, ax = plt.subplots(len(picks), 4, figsize=(12, 3.1 * len(picks)))
ax = ax.reshape(len(picks), 4)
for r, ((iou, im, gt, pr, at), tag) in enumerate(zip(picks, tags)):
    ax[r, 0].imshow(_to_hwc_uint8(im))
    ax[r, 1].imshow(colorize_mask(gt.numpy()))
    ax[r, 2].imshow(colorize_mask(pr.numpy()))
    ax[r, 3].imshow(overlay_csp_attention(im, at) if at is not None else _to_hwc_uint8(im))
    for c in range(4):
        ax[r, c].axis("off")
        if r == 0: ax[r, c].set_title(cols[c], fontsize=12)
    ax[r, 0].text(-0.09, 0.5, f"{tag}\nIoU={iou:.2f}", transform=ax[r, 0].transAxes,
                  rotation=90, va="center", ha="center", fontsize=11, fontweight="bold",
                  color=("#009E73" if tag == "Success" else "#D55E00"))
    # scale bar: Sentinel-2 is 10 m/px, so 100 px = 1 km
    H = im.shape[-1]
    ax[r, 0].add_patch(Rectangle((0.06 * H, 0.90 * H), 100, 0.02 * H, color="white", ec="k"))
    ax[r, 0].text(0.06 * H, 0.86 * H, "1 km", color="white", fontsize=8, fontweight="bold")
plt.tight_layout()
fig.savefig(f"{PAPER_FIG}/qualitative.pdf", bbox_inches="tight", dpi=300)
fig.savefig(f"{PAPER_FIG}/qualitative.png", bbox_inches="tight", dpi=300)
plt.show()
print("saved -> qualitative.pdf/.png (success/failure rows + scale bars, test split) in", PAPER_FIG)


## 14. Leave-one-region-out generalization (headline protocol)

Hold out whole continents from training and test on them — the publishable cross-region
result. Set `holdout_regions`, rebuild loaders (train/val now exclude those regions, and an
`ood` split appears), retrain, and compare in-region vs out-of-region.


In [ ]:
# cfg.data.holdout_regions = ["Asia", "Oceania"]
# loaders = build_dataloaders(cfg, splits=("train", "val", "ood"))
# model = build_model(cfg); trainer = Trainer(model, loaders, cfg); trainer.train()
# in_region  = evaluate(model, loaders["val"], cfg, device)["miou"]
# out_region = evaluate(model, loaders["ood"], cfg, device)["miou"]
# print(f"in-region mIoU={in_region:.3f}  out-of-region mIoU={out_region:.3f}  "
#       f"drop={in_region-out_region:.3f}")


## 15. Ablations + baselines — FULL-budget paper table

Each of the 8 variants gets **its own cell** (15.1–15.8) so you can run them one at a time,
watch each, and resume individually. They all append to one shared results JSON on Drive;
**15.9** renders the combined table.

**Full budget:** `FULL_EPOCHS = 40`, full data (`subset = None`). Each variant is ~1.5–2 h on
an A100, so ~12–16 h total — run a few per session; each is **resumable** (re-run its cell and
it continues from `last.pt`, or skips if already recorded). Lower `FULL_EPOCHS` for a faster
(weaker) pass.


In [ ]:
from spearnet.config import load_config
from spearnet.engine import run_one_experiment, run_seeds, comparison_status
import os

# ---- FULL budget for the paper table ----
FULL_EPOCHS       = 40
FULL_SUBSET_TRAIN = None      # None = all training chips
FULL_SUBSET_VAL   = None      # None = full val (representative; shuffled before any cap)

base = load_config(CONFIG)            # CONFIG from cell 7 (configs/v4_scale3.yaml)
base.data.source = "mining"
base.data.chips_root = CHIPS_DIR
base.data.manifest   = f"{CHIPS_DIR}/manifest.csv"
base.data.num_workers = 2

RESULTS = f"/content/drive/MyDrive/spearnet_runs/comparison_full_{base.data.task}.json"
RUNS    = f"/content/drive/MyDrive/spearnet_runs/compare_full_{base.data.task}"

EXPERIMENTS = {
    # ----- ablations (SPEAR-Net backbone, vary one thing at a time) -----
    "A1 backbone + CE (no prior)": {"model.csp_mode": "none", "model.use_edge_head": False,
        "loss.w_ce": 1.0, "loss.w_focal": 0.0, "loss.w_dice": 0.0, "loss.w_tversky": 0.0,
        "loss.w_boundary": 0.0, "loss.w_edge": 0.0, "loss.auto_class_weights": False},
    "A2 + recall loss": {"model.csp_mode": "none", "model.use_edge_head": False,
        "loss.w_boundary": 0.0, "loss.w_edge": 0.0},
    "A3 + PISP concat": {"model.csp_mode": "concat", "model.use_edge_head": False,
        "loss.w_boundary": 0.0, "loss.w_edge": 0.0},
    "A4 SPEAR-Net (PISP gate + edge)": {"model.csp_mode": "gate", "model.use_edge_head": True},
    "A5 RGB prior (csp, 3-band)": {"model.prior_type": "csp", "data.bands": 3,
        "model.csp_mode": "gate", "model.use_edge_head": True},
    # ----- baselines (segmentation-models-pytorch), same data + loss -----
    "B1 U-Net (resnet34)": {"model.name": "unet", "model.backbone": "resnet34"},
    "B2 DeepLabV3+ (resnet34)": {"model.name": "deeplabv3p", "model.backbone": "resnet34"},
    "B3 U-Net++ (resnet34)": {"model.name": "unetpp", "model.backbone": "resnet34"},
}
print("Setup ready. Run cells 15.1–15.8 (one variant each), then 15.9 for the table.")
print("Variants:", list(EXPERIMENTS))


### 15.0 — Status (run after any crash to see what to resume)

Shows each variant as **done** / **resumable @ epoch N** / **not started**, so you know
exactly which cells to re-run. (Re-running a variant cell resumes it automatically.)


In [ ]:
import pandas as pd
status = comparison_status(EXPERIMENTS, RESULTS, RUNS, epochs=FULL_EPOCHS)
display(pd.DataFrame(status))
todo = [s["name"] for s in status if s["status"] != "done"]
print(f"{len(EXPERIMENTS)-len(todo)}/{len(EXPERIMENTS)} done. Still to run/resume: {todo}")


### 15.1 — A1 backbone + CE (no prior)

In [ ]:
run_one_experiment(base, 'A1 backbone + CE (no prior)', EXPERIMENTS['A1 backbone + CE (no prior)'], RESULTS, RUNS,
                   epochs=FULL_EPOCHS, subset_train=FULL_SUBSET_TRAIN,
                   subset_val=FULL_SUBSET_VAL)


### 15.2 — A2 + recall loss

In [ ]:
run_one_experiment(base, 'A2 + recall loss', EXPERIMENTS['A2 + recall loss'], RESULTS, RUNS,
                   epochs=FULL_EPOCHS, subset_train=FULL_SUBSET_TRAIN,
                   subset_val=FULL_SUBSET_VAL)


### 15.3 — A3 + PISP concat

In [ ]:
run_one_experiment(base, 'A3 + PISP concat', EXPERIMENTS['A3 + PISP concat'], RESULTS, RUNS,
                   epochs=FULL_EPOCHS, subset_train=FULL_SUBSET_TRAIN,
                   subset_val=FULL_SUBSET_VAL)


### 15.4 — A4 SPEAR-Net (PISP gate + edge)

In [ ]:
run_one_experiment(base, 'A4 SPEAR-Net (PISP gate + edge)', EXPERIMENTS['A4 SPEAR-Net (PISP gate + edge)'], RESULTS, RUNS,
                   epochs=FULL_EPOCHS, subset_train=FULL_SUBSET_TRAIN,
                   subset_val=FULL_SUBSET_VAL)


### 15.5 — A5 RGB prior (csp, 3-band)

In [ ]:
run_one_experiment(base, 'A5 RGB prior (csp, 3-band)', EXPERIMENTS['A5 RGB prior (csp, 3-band)'], RESULTS, RUNS,
                   epochs=FULL_EPOCHS, subset_train=FULL_SUBSET_TRAIN,
                   subset_val=FULL_SUBSET_VAL)


### 15.6 — B1 U-Net (resnet34)

In [ ]:
run_one_experiment(base, 'B1 U-Net (resnet34)', EXPERIMENTS['B1 U-Net (resnet34)'], RESULTS, RUNS,
                   epochs=FULL_EPOCHS, subset_train=FULL_SUBSET_TRAIN,
                   subset_val=FULL_SUBSET_VAL)


### 15.7 — B2 DeepLabV3+ (resnet34)

In [ ]:
run_one_experiment(base, 'B2 DeepLabV3+ (resnet34)', EXPERIMENTS['B2 DeepLabV3+ (resnet34)'], RESULTS, RUNS,
                   epochs=FULL_EPOCHS, subset_train=FULL_SUBSET_TRAIN,
                   subset_val=FULL_SUBSET_VAL)


### 15.8 — B3 U-Net++ (resnet34)

In [ ]:
run_one_experiment(base, 'B3 U-Net++ (resnet34)', EXPERIMENTS['B3 U-Net++ (resnet34)'], RESULTS, RUNS,
                   epochs=FULL_EPOCHS, subset_train=FULL_SUBSET_TRAIN,
                   subset_val=FULL_SUBSET_VAL)


### 15.9 — Show the combined comparison table

In [ ]:
import pandas as pd, json
rows = json.load(open(RESULTS)) if os.path.exists(RESULTS) else []
df = pd.DataFrame(rows)
cols = [c for c in ["name","model","prior","csp_mode","bands","mIoU","mRecall",
                    "IoU_background","IoU_artisanal","IoU_industrial","IoU_mining",
                    "params_M","gflops","epochs","error"] if c in df.columns]
df = df[cols].sort_values("mIoU", ascending=False, na_position="last") if "mIoU" in df.columns else df
import IPython.display as D; D.display(df)
df.to_markdown(RESULTS.replace(".json", ".md"), index=False)
print(f"{len(rows)}/8 variants done. Saved table -> {RESULTS.replace('.json', '.md')}")


## 16. Robustness — 3 seeds (mean ± std) for the headline model

Re-trains the chosen model under 3 seeds and reports mean ± std (the numbers you put in the
paper). Defaults to **A4** (full SPEAR-Net); switch `SEED_VARIANT` to `"A3 + PISP concat"` if
concat wins the table. Resumable (finished seeds are skipped).


In [ ]:
SEED_VARIANT = "A4 SPEAR-Net (PISP gate + edge)"   # or "A3 + PISP concat"
SEEDS = [42, 7, 123]

seed_out = run_seeds(
    base, SEEDS,
    results_path=f"/content/drive/MyDrive/spearnet_runs/seeds_{base.data.task}.json",
    runs_root=f"/content/drive/MyDrive/spearnet_runs/seeds_{base.data.task}",
    overrides=EXPERIMENTS[SEED_VARIANT],
    epochs=FULL_EPOCHS, subset_train=FULL_SUBSET_TRAIN, subset_val=FULL_SUBSET_VAL,
)
import pandas as pd
display(pd.DataFrame(seed_out["rows"]))
print("\nmean ± std:")
for k, v in seed_out["summary"].items():
    print(f"  {k:>16}: {v['mean']:.4f} ± {v['std']:.4f}  (n={v['n']})")


## R. Reviewer-response evidence (Phase A + B)

Runs the diagnostics and test-split evaluation the manuscript's Tables 1–5 must be rebuilt
from. **Prerequisites:** run cell 15 (setup: defines `base`, `EXPERIMENTS`, `RUNS`), have the
chips restored (4b) and the 8 comparison checkpoints trained (15.1–15.8). Everything is saved
to `MyDrive/spearnet_runs/` — paste the printed tables back for the manuscript rewrite.


### R1 — Dataset diagnostics (Phase A: fixes Table 1 and §4.4)

In [ ]:
import glob
GPKG = (glob.glob("mine_data/*.gpkg") or ["mine_data/mining_area_data.gpkg"])[0]
!python scripts/diagnostics.py --gpkg "{GPKG}" --manifest "{CHIPS_DIR}/manifest.csv" \
    --out /content/drive/MyDrive/spearnet_runs/diagnostics.json


### R2 — Test-split metrics for all 8 variants (Phase B1–B5)

Per-class IoU / precision / recall / F1, the 3×3 confusion matrix, foreground-mIoU, and
**area-stratified recall for every model including the baselines** (item B5 — the headline).


In [ ]:
# Prereqs: run '4. Mount Drive', '4b. Restore chips', and the '15. Ablations + baselines'
# SETUP cell (defines base / RUNS / EXPERIMENTS). No training needed here -- this only
# evaluates the checkpoints already on Drive, on the TEST split.
from spearnet.config import override_config
from spearnet.data import build_dataloaders
from spearnet.models import build_model
from spearnet.engine import evaluate
from spearnet.engine.compare import _safe
import os, json, torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

TEST = {}
for name, ov in EXPERIMENTS.items():
    ckpt = os.path.join(RUNS, _safe(name), "best.pt")
    if not os.path.exists(ckpt):
        print("MISSING checkpoint ->", name, "(", ckpt, ") — will report as limitation")
        continue
    c = override_config(base, ov); c.data.subset_test = None
    ld = build_dataloaders(c, splits=("test",))
    m = build_model(c).to(device)
    m.load_state_dict(torch.load(ckpt, map_location=device)["model"])
    res = evaluate(m, ld["test"], c, device, compute_area_stratified=True)
    TEST[name] = res
    pc = res["per_class"]
    ar = res.get("area_stratified_recall", {})
    print(f"{name:<34} mIoU={res['miou']:.3f} fgmIoU={res.get('fg_miou',0):.3f} "
          f"R(art)={pc.get('artisanal',{}).get('recall',0):.3f} "
          f"R(ind)={pc.get('industrial',{}).get('recall',0):.3f} | "
          f"area R: small={ar.get('small','-')} med={ar.get('medium','-')} large={ar.get('large','-')}")
    json.dump(res, open(f"/content/drive/MyDrive/spearnet_runs/test_{_safe(name)}.json","w"), indent=2)

json.dump({k:{kk:vv for kk,vv in v.items() if kk!='confusion_matrix'} for k,v in TEST.items()},
          open("/content/drive/MyDrive/spearnet_runs/test_all.json","w"), indent=2, default=str)
print("\nSaved per-model test metrics + test_all.json to Drive.")
print("Confusion matrices are inside each test_<name>.json (key 'confusion_matrix').")


### R3 — Gate analysis + efficiency profile (B6–B9)

In [ ]:
# Prereqs: run '4. Mount Drive' and the '15. Ablations + baselines' SETUP cell
# (defines RUNS / CHIPS_DIR). No training needed.
from spearnet.engine.compare import _safe
GATE = f"{RUNS}/{_safe('A4 SPEAR-Net (PISP gate + edge)')}/best.pt"
# B7 learned alpha_s + B8 PISP faithfulness ROC-AUC (gate model, test split)
!python scripts/analyze_model.py --config configs/v4_scale3.yaml --checkpoint "{GATE}" \
    --split test --chips-root "{CHIPS_DIR}" \
    --out /content/drive/MyDrive/spearnet_runs/analysis_gate.json
# B6 throughput/VRAM (GPU+CPU) + B9 hours-per-Sentinel-2-tile + honest FLOP ratios
!python scripts/benchmark.py --config configs/v4_scale3.yaml \
    --models spearnet unet deeplabv3p unetpp --cpu \
    --out /content/drive/MyDrive/spearnet_runs/benchmark.json


## 17. Generate the paper figures (no re-training)

Builds the four data figures for the manuscript from results that already exist:
dataset world map, mIoU-vs-parameters scatter (the headline figure), ablation bars, and
area-stratified recall bars. Colorblind-safe palette; PNG (300 dpi) + vector PDF are
written to Drive — download and drop into Overleaf `figures/`.


In [ ]:
# Area-stratified values from your evaluation cell (update if you re-evaluate):
AREA        = [0.537, 0.729, 0.838]   # gate-model small/medium/large recall
AREA_COUNTS = [82, 59, 37]   # corrected component counts (small/medium/large)

FIG_DIR = "/content/drive/MyDrive/spearnet_runs/paper_figures"
!pip -q install geopandas mapclassify 2>/dev/null
!python scripts/make_figures.py \
    --results "/content/drive/MyDrive/spearnet_runs/comparison_full_scale3.json" \
    --gpkg "mine_data/mining_area_data.gpkg" \
    --area {AREA[0]} {AREA[1]} {AREA[2]} \
    --area-counts {AREA_COUNTS[0]} {AREA_COUNTS[1]} {AREA_COUNTS[2]} \
    --out "{FIG_DIR}"

import glob
from IPython.display import Image as IPImage, display
for p in sorted(glob.glob(f"{FIG_DIR}/*.png")):
    print(p); display(IPImage(p, width=640))


## 17b. Architecture-diagram thumbnails

Exports the real image slots used by the TikZ architecture figure
(`paper/figures/architecture.tex`): the S2 input patch, the four PISP index maps, the
learned attention map, the prediction, and the edge map — from one mining-containing
validation chip. Download them into `paper/figures/` (same names) and recompile
`architecture.tex`.


In [ ]:
import os, numpy as np, torch, matplotlib.pyplot as plt
from spearnet.data import compute_priors
from spearnet.data.priors import band_index
from spearnet.utils.viz import colorize_mask, _to_hwc_uint8

TH = "/content/drive/MyDrive/spearnet_runs/paper_figures"; os.makedirs(TH, exist_ok=True)
model.eval(); device = torch.device(cfg.run.device)
with torch.no_grad():
    vb = next(iter(loaders["val"]))
    j = next((i for i in range(vb["mask"].shape[0]) if (vb["mask"][i] > 0).any()), 0)
    x = vb["image"][j]
    out = model(x.unsqueeze(0).to(device))
    pred = out["logits"].argmax(1)[0].cpu().numpy()
    attn = out["attn"][0, 0].cpu().numpy() if "attn" in out else None
    edge = torch.sigmoid(out["edge"][0, 0]).cpu().numpy() if "edge" in out else None

plt.imsave(f"{TH}/input_patch.png", _to_hwc_uint8(x))
idxm = band_index(cfg.data.band_order)
pri = compute_priors(x.unsqueeze(0), cfg.model.prior_type, idxm)[0].numpy()
for a, n, cm in zip(pri, ["ndvi","mndwi","ndti","bsi"], ["RdYlGn","YlGnBu","viridis","inferno"]):
    plt.imsave(f"{TH}/{n}.png", a, cmap=cm)
if attn is not None: plt.imsave(f"{TH}/attention.png", attn, cmap="hot")
plt.imsave(f"{TH}/prediction.png", colorize_mask(pred))
if edge is not None: plt.imsave(f"{TH}/edge.png", edge, cmap="gray")

import glob
from IPython.display import Image as IPImage, display
print("saved ->", TH)
for p in sorted(glob.glob(f"{TH}/*.png")):
    print(p); display(IPImage(p, width=140))


## 18. Next steps

* Run the table for `CONFIG = configs/v4_binary.yaml` too (the detector result).
* Generalization headline: rotate `data.holdout_regions` across continents (cell 14) and
  report the mean in-region vs out-of-region gap.
* Pick the final model from 15.9 (best mIoU-vs-params trade-off) and report its seeds
  (cell 16) as the headline numbers.
